In [329]:
# Objective:
# To identify structured financial behavior patterns through layered analysis
# combining anomaly detection, timeline reconstruction, and relationship mapping.

import sys
from pathlib import Path

# Proje root'unu bul
CURRENT = Path.cwd().resolve()

if (CURRENT / "src").exists():
    ROOT = CURRENT
elif (CURRENT.parent / "src").exists():
    ROOT = CURRENT.parent
else:
    raise FileNotFoundError("Project root not found")

# sys.path'e ekle
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"[INFO] Project root set to: {ROOT}")

[INFO] Project root set to: C:\Users\adria\OneDrive\Desktop\Operation Cold Ledger


In [330]:
import sys
from pathlib import Path
import importlib

CURRENT = Path.cwd().resolve()

if (CURRENT / "src").exists():
    ROOT = CURRENT
elif (CURRENT.parent / "src").exists():
    ROOT = CURRENT.parent
else:
    raise FileNotFoundError("Could not find project root containing 'src' folder.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.load_data as load_data_module
import src.anomaly_detection as anomaly_module
import src.risk_scoring as risk_module
import src.timeline_builder as timeline_module

importlib.reload(load_data_module)
importlib.reload(anomaly_module)
importlib.reload(risk_module)
importlib.reload(timeline_module)

from src.load_data import load_transactions
from src.anomaly_detection import run_all_anomaly_checks
from src.risk_scoring import score_account_risk
from src.timeline_builder import build_full_timeline, build_account_timeline

print("Imports successful")

Imports successful


In [331]:
data_path = ROOT / "data" / "raw" / "synthetic_transactions.csv"

print("Data path:", data_path)
print("File exists:", data_path.exists())

df = load_transactions(str(data_path))
print("Data loaded:", df.shape)

df.head()

Data path: C:\Users\adria\OneDrive\Desktop\Operation Cold Ledger\data\raw\synthetic_transactions.csv
File exists: True
Data loaded: (12104, 18)


,transaction_id,account_id,timestamp,amount,currency,transaction_type,counterparty,country,channel,device_id,ip_address,status,account_status_change,session_id,geo_velocity_km_h,risk_score,is_flagged,notes
0,TXN-000003,ACC-1001,2024-01-01 00:00:00,7711,USD,outbound,CTR-137,FR,mobile,DEV-2519,98.25.92.217,completed,email_changed,SES-10329,1349,95,True,anomalous burst
1,TXN-000004,ACC-1001,2024-01-01 00:03:00,11803,USD,outbound,CTR-206,FR,mobile,DEV-2519,98.25.92.217,blocked,none,SES-6007,1391,90,True,anomalous burst
2,TXN-000005,ACC-1001,2024-01-01 00:06:00,10733,USD,outbound,CTR-885,FR,mobile,DEV-2519,98.25.92.217,blocked,none,SES-13239,881,90,True,anomalous burst
3,TXN-000006,ACC-1001,2024-01-01 00:06:00,11554,USD,outbound,CTR-864,FR,mobile,DEV-2519,98.25.92.217,completed,none,SES-21320,1450,90,True,anomalous burst
4,TXN-000007,ACC-1001,2024-01-01 00:08:00,12820,USD,outbound,CTR-283,FR,web,DEV-2519,98.25.92.217,blocked,none,SES-79841,1486,90,True,anomalous burst


In [332]:
anomalies = run_all_anomaly_checks(df)
print("Anomalies shape:", anomalies.shape)
anomalies.head(10)

Anomalies shape: (1589, 5)


,account_id,transaction_id,anomaly_type,indicator_strength,observation
0,ACC-1001,TXN-000003,cross_border_change,high,Country pattern shifted across 6 locations: FR...
1,ACC-1001,TXN-000003,night_activity,low,Transaction observed at 2024-01-01 00:00:00
2,ACC-1001,TXN-000004,night_activity,low,Transaction observed at 2024-01-01 00:03:00
3,ACC-1001,TXN-000005,night_activity,low,Transaction observed at 2024-01-01 00:06:00
4,ACC-1001,TXN-000006,night_activity,low,Transaction observed at 2024-01-01 00:06:00
5,ACC-1001,TXN-000007,night_activity,low,Transaction observed at 2024-01-01 00:08:00
6,ACC-1001,TXN-000062,night_activity,low,Transaction observed at 2024-01-05 02:00:00
7,ACC-1001,TXN-000063,night_activity,low,Transaction observed at 2024-01-05 02:02:00
8,ACC-1001,TXN-000065,night_activity,low,Transaction observed at 2024-01-05 02:03:00
9,ACC-1001,TXN-000066,night_activity,low,Transaction observed at 2024-01-05 02:04:00


In [333]:
risk = score_account_risk(anomalies)
print("Risk shape:", risk.shape)
risk.head(10)

Risk shape: (5, 4)


,account_id,risk_score,risk_level,triggered_indicators
0,ACC-1001,1000,high,"[cross_border_change, night_activity, post_acc..."
4,ACC-9001,562,high,"[cross_border_change, night_activity, post_acc..."
3,ACC-7788,228,high,"[cross_border_change, night_activity, post_acc..."
1,ACC-2044,218,high,"[cross_border_change, night_activity, post_acc..."
2,ACC-3320,174,high,"[cross_border_change, night_activity, post_acc..."


In [334]:
print("=== ANALYST NOTE ===")
print(f"Total anomalies detected: {len(anomalies)}")
print(f"Accounts analyzed: {df['account_id'].nunique()}")
print(f"High risk accounts: {(risk['risk_level'] == 'high').sum()}")
print(f"Elevated risk accounts: {(risk['risk_level'] == 'elevated').sum()}")

=== ANALYST NOTE ===
Total anomalies detected: 1589
Accounts analyzed: 5
High risk accounts: 5
Elevated risk accounts: 0


In [335]:
data_path = ROOT / "data" / "raw" / "synthetic_transactions.csv"

df = load_transactions(str(data_path))
anomalies = run_all_anomaly_checks(df)
risk = score_account_risk(anomalies)
timeline = build_full_timeline(df)

print("df:", df.shape)
print("anomalies:", anomalies.shape)
print("risk:", risk.shape)
print("timeline:", timeline.shape)

display(anomalies.head())
display(risk.head())
display(timeline.head())

df: (12104, 18)
anomalies: (1589, 5)
risk: (5, 4)
timeline: (2831, 5)


,account_id,transaction_id,anomaly_type,indicator_strength,observation
0,ACC-1001,TXN-000003,cross_border_change,high,Country pattern shifted across 6 locations: FR...
1,ACC-1001,TXN-000003,night_activity,low,Transaction observed at 2024-01-01 00:00:00
2,ACC-1001,TXN-000004,night_activity,low,Transaction observed at 2024-01-01 00:03:00
3,ACC-1001,TXN-000005,night_activity,low,Transaction observed at 2024-01-01 00:06:00
4,ACC-1001,TXN-000006,night_activity,low,Transaction observed at 2024-01-01 00:06:00


,account_id,risk_score,risk_level,triggered_indicators
0,ACC-1001,1000,high,"[cross_border_change, night_activity, post_acc..."
4,ACC-9001,562,high,"[cross_border_change, night_activity, post_acc..."
3,ACC-7788,228,high,"[cross_border_change, night_activity, post_acc..."
1,ACC-2044,218,high,"[cross_border_change, night_activity, post_acc..."
2,ACC-3320,174,high,"[cross_border_change, night_activity, post_acc..."


,account_id,timestamp,event_type,transaction_id,event_summary
0,ACC-1001,2024-01-01 00:00:00,account_change,TXN-000003,Account event detected: email_changed
1,ACC-1001,2024-01-01 00:00:00,night_activity,TXN-000003,Night transaction via mobile from FR
2,ACC-1001,2024-01-01 00:03:00,night_activity,TXN-000004,Night transaction via mobile from FR
3,ACC-1001,2024-01-01 00:03:00,high_value_transaction,TXN-000004,High-value transaction observed: 11803
4,ACC-1001,2024-01-01 00:06:00,night_activity,TXN-000005,Night transaction via mobile from FR


In [336]:
from src.clean_data import clean_transaction_data

df_clean = clean_transaction_data(df)

print(df_clean.shape)
df_clean.head()

[CLEANING] Removed 0 duplicate rows
(12104, 25)


,transaction_id,account_id,timestamp,amount,currency,transaction_type,counterparty,country,channel,device_id,...,risk_score,is_flagged,notes,hour,is_night,amount_abs,is_outlier,missing_timestamp,missing_amount,invalid_amount
0,TXN-000003,ACC-1001,2024-01-01 00:00:00,7711,usd,outbound,CTR-137,fr,mobile,DEV-2519,...,95,True,anomalous burst,0,1,7711,False,False,False,False
1,TXN-000004,ACC-1001,2024-01-01 00:03:00,11803,usd,outbound,CTR-206,fr,mobile,DEV-2519,...,90,True,anomalous burst,0,1,11803,False,False,False,False
2,TXN-000005,ACC-1001,2024-01-01 00:06:00,10733,usd,outbound,CTR-885,fr,mobile,DEV-2519,...,90,True,anomalous burst,0,1,10733,False,False,False,False
3,TXN-000006,ACC-1001,2024-01-01 00:06:00,11554,usd,outbound,CTR-864,fr,mobile,DEV-2519,...,90,True,anomalous burst,0,1,11554,False,False,False,False
4,TXN-000007,ACC-1001,2024-01-01 00:08:00,12820,usd,outbound,CTR-283,fr,web,DEV-2519,...,90,True,anomalous burst,0,1,12820,False,False,False,False


In [337]:
df_clean["is_outlier"].value_counts()
df_clean["missing_amount"].sum()
df_clean["invalid_amount"].sum()

np.int64(0)

In [338]:
import importlib
import src.network_analysis as network_analysis_module

importlib.reload(network_analysis_module)
from src.network_analysis import (
    summarize_counterparties,
    find_top_counterparties,
    summarize_country_exposure,
    build_account_counterparty_matrix,
    detect_concentrated_counterparty_risk,
)

Analysis cells

In [339]:
counterparty_summary = summarize_counterparties(df_clean)
counterparty_summary.head(10)

,account_id,unique_counterparties,total_transactions
0,ACC-9001,979,4033
1,ACC-1001,955,3667
2,ACC-2044,840,1975
3,ACC-3320,791,1579
4,ACC-7788,577,850


In [340]:
top_counterparties = find_top_counterparties(df_clean, top_n=10)
top_counterparties

,counterparty,transaction_count
0,CTR-116,23
1,CTR-711,23
2,CTR-79,22
3,CTR-315,22
4,CTR-592,22
5,CTR-795,21
6,CTR-358,21
7,CTR-508,21
8,CTR-420,21
9,CTR-504,20


In [341]:
country_exposure = summarize_country_exposure(df_clean)
country_exposure.head(10)

,account_id,unique_countries,total_transactions
0,ACC-9001,6,4033
1,ACC-1001,6,3667
2,ACC-2044,6,1975
3,ACC-3320,6,1579
4,ACC-7788,6,850


In [342]:
relationship_matrix = build_account_counterparty_matrix(df_clean)
relationship_matrix.head()

counterparty,CTR-10,CTR-100,CTR-101,CTR-102,CTR-103,CTR-104,CTR-105,CTR-106,CTR-107,CTR-108,...,CTR-990,CTR-991,CTR-992,CTR-993,CTR-994,CTR-995,CTR-996,CTR-997,CTR-998,CTR-999
account_id,,,,,,,,,,,,,,,,,,,,,
ACC-1001,2,3,6,7,3,4,2,5,1,1,...,2,3,3,3,1,2,3,2,5,5
ACC-2044,2,3,2,2,3,1,2,0,1,2,...,6,4,4,2,2,0,1,3,0,0
ACC-3320,2,2,0,3,1,2,1,0,1,2,...,0,1,2,1,1,1,5,3,1,2
ACC-7788,0,2,1,0,0,0,2,4,0,0,...,0,0,0,1,1,1,0,1,1,0
ACC-9001,2,1,2,6,5,4,3,3,4,3,...,4,3,1,4,2,3,3,5,5,10


In [343]:
concentration_risk = detect_concentrated_counterparty_risk(df_clean, threshold=0.5)
print(concentration_risk.shape)
concentration_risk.head()

(0, 4)


,account_id,dominant_counterparty,dominant_share,total_transactions
